# Experiment 8 — Denoising Autoencoder on Noisy MNIST

**Module 4 · Topic: Autoencoder · CO5**

### Problem Statement
> Build and train a **denoising autoencoder** to reconstruct corrupted images
> (e.g. noisy MNIST digits).

### What this notebook covers
1. MNIST loading + five kinds of corruption (Gaussian, salt-&-pepper, speckle, block masking, blur).
2. A **fully-connected (dense) DAE** — the classic formulation.
3. A **convolutional DAE** with skip-free encoder/decoder — the one that actually works well.
4. Comparison against classical denoisers (median filter, Gaussian blur) as a sanity baseline.
5. Quantitative evaluation: **MSE, PSNR, SSIM**, plus noise-level and bottleneck-size sweeps.
6. Latent-space analysis (t-SNE, interpolation), and a **downstream utility test**: does denoising
   actually recover classifier accuracy on corrupted digits?
7. Bonus: the same DAE used for **anomaly detection** via reconstruction error.

### How to run
`Runtime -> Change runtime type -> **T4 GPU**`, then `Runtime -> Run all` (~12–18 min).

## 0. Setup

In [ ]:
import os, time, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable the T4 GPU via Runtime -> Change runtime type")

## 1. Data and corruption models

An autoencoder is **self-supervised**: the label is the input itself. A *denoising* autoencoder
changes one thing — the input is corrupted, the **target stays clean**:

$$\hat{x} = g_\theta(f_\phi(\tilde{x})), \qquad \mathcal{L} = \| \hat{x} - x \|^2, \qquad \tilde{x} \sim q(\tilde{x}\,|\,x)$$

Forcing the network to undo `q` prevents it from learning the identity function and pushes it to
capture the **data manifold** — the structure that distinguishes a digit from noise.

In [ ]:
tf = transforms.ToTensor()
train_full = datasets.MNIST("./data", train=True,  download=True, transform=tf)
test_set   = datasets.MNIST("./data", train=False, download=True, transform=tf)
train_set, val_set = random_split(train_full, [54000, 6000],
                                  generator=torch.Generator().manual_seed(SEED))
print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)}")

In [ ]:
def add_noise(x, kind="gaussian", level=0.5, generator=None):
    # x: float tensor in [0,1], shape (B,1,28,28) or (1,28,28)
    g = generator
    if kind == "gaussian":
        out = x + torch.randn(x.shape, generator=g, device=x.device) * level
    elif kind == "salt_pepper":
        r = torch.rand(x.shape, generator=g, device=x.device)
        out = x.clone()
        out[r < level / 2] = 0.0
        out[r > 1 - level / 2] = 1.0
    elif kind == "speckle":
        out = x * (1 + torch.randn(x.shape, generator=g, device=x.device) * level)
    elif kind == "block":
        out = x.clone()
        b = out.shape[0] if out.dim() == 4 else 1
        view = out if out.dim() == 4 else out.unsqueeze(0)
        size = max(4, int(14 * level))
        for i in range(b):
            for _ in range(2):
                r0 = random.randint(0, 28 - size); c0 = random.randint(0, 28 - size)
                view[i, :, r0:r0+size, c0:c0+size] = torch.rand(1).item()
        out = view if x.dim() == 4 else view.squeeze(0)
    elif kind == "blur":
        k = torch.ones(1, 1, 3, 3, device=x.device) / 9.0
        v = x if x.dim() == 4 else x.unsqueeze(0)
        v = F.conv2d(v, k, padding=1)
        v = v + torch.randn(v.shape, generator=g, device=v.device) * (level * 0.3)
        out = v if x.dim() == 4 else v.squeeze(0)
    else:
        raise ValueError(kind)
    return out.clamp(0.0, 1.0)

NOISE_KINDS = ["gaussian", "salt_pepper", "speckle", "block", "blur"]

# Visualise every corruption model
clean = torch.stack([test_set[i][0] for i in range(8)])
fig, axes = plt.subplots(len(NOISE_KINDS) + 1, 8, figsize=(12, 9))
for j in range(8):
    axes[0, j].imshow(clean[j, 0], cmap="gray", vmin=0, vmax=1); axes[0, j].axis("off")
axes[0, 0].set_ylabel("clean")
axes[0, 0].set_title("clean", loc="left", fontsize=9)
for i, kind in enumerate(NOISE_KINDS, start=1):
    torch.manual_seed(SEED)
    noisy = add_noise(clean, kind, 0.5)
    for j in range(8):
        axes[i, j].imshow(noisy[j, 0], cmap="gray", vmin=0, vmax=1); axes[i, j].axis("off")
    axes[i, 0].set_title(kind, loc="left", fontsize=9)
plt.suptitle("Corruption models q(x_tilde | x)"); plt.tight_layout(); plt.show()

In [ ]:
class NoisyMNIST(Dataset):
    # returns (corrupted, clean, label); noise is re-sampled every epoch for the train split
    def __init__(self, base, kind="gaussian", level=0.5, fixed=False, seed=0):
        self.base, self.kind, self.level, self.fixed, self.seed = base, kind, level, fixed, seed
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        if self.fixed:
            g = torch.Generator().manual_seed(self.seed + i)
            random.seed(self.seed + i)
        else:
            g = None
        return add_noise(x, self.kind, self.level, generator=g), x, y

NOISE_KIND, NOISE_LEVEL, BATCH = "gaussian", 0.5, 256
tr_dl = DataLoader(NoisyMNIST(train_set, NOISE_KIND, NOISE_LEVEL), batch_size=BATCH,
                   shuffle=True, num_workers=2, pin_memory=True)
va_dl = DataLoader(NoisyMNIST(val_set,  NOISE_KIND, NOISE_LEVEL, fixed=True, seed=1),
                   batch_size=512, num_workers=2, pin_memory=True)
te_dl = DataLoader(NoisyMNIST(test_set, NOISE_KIND, NOISE_LEVEL, fixed=True, seed=2),
                   batch_size=512, num_workers=2, pin_memory=True)
xb, yb, lb = next(iter(tr_dl))
print("batch:", xb.shape, yb.shape, "| input range", float(xb.min()), float(xb.max()))

## 2. Model A — Dense (fully-connected) denoising autoencoder

`784 -> 512 -> 256 -> 64 (bottleneck) -> 256 -> 512 -> 784`

The **bottleneck** of 64 units is 12x smaller than the input, so the network cannot copy its
input through; it must compress to the informative factors (stroke position, thickness, slant).

In [ ]:
class DenseDAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, 784), nn.Sigmoid(),
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z).view(-1, 1, 28, 28)
    def encode(self, x):
        return self.encoder(x)

print("DenseDAE params:", f"{sum(p.numel() for p in DenseDAE().parameters()):,}")

## 3. Model B — Convolutional denoising autoencoder

Convolutions respect the 2-D structure that flattening destroys, and weight sharing makes the
model far more parameter-efficient. The decoder upsamples with `ConvTranspose2d`.

`28x28x1 -> 14x14x32 -> 7x7x64 -> 4x4x128 (latent) -> 7x7x64 -> 14x14x32 -> 28x28x1`

In [ ]:
class ConvDAE(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.encoder = nn.Sequential(
            nn.Conv2d(1, b, 3, stride=2, padding=1),      nn.BatchNorm2d(b),     nn.ReLU(inplace=True),   # 14
            nn.Conv2d(b, 2*b, 3, stride=2, padding=1),    nn.BatchNorm2d(2*b),   nn.ReLU(inplace=True),   # 7
            nn.Conv2d(2*b, 4*b, 3, stride=2, padding=1),  nn.BatchNorm2d(4*b),   nn.ReLU(inplace=True),   # 4
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(4*b, 2*b, 3, stride=2, padding=1, output_padding=0),  # 7
            nn.BatchNorm2d(2*b), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(2*b, b, 3, stride=2, padding=1, output_padding=1),    # 14
            nn.BatchNorm2d(b), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(b, 1, 3, stride=2, padding=1, output_padding=1),      # 28
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))
    def encode(self, x):
        return self.encoder(x).flatten(1)

m = ConvDAE()
print("ConvDAE params:", f"{sum(p.numel() for p in m.parameters()):,}")
with torch.no_grad():
    t = torch.randn(2, 1, 28, 28)
    print("encoder output:", m.encoder(t).shape, "-> reconstruction:", m(t).shape)

## 4. Training

In [ ]:
def psnr(x, y, eps=1e-10):
    mse = F.mse_loss(x, y, reduction="none").flatten(1).mean(1)
    return (10 * torch.log10(1.0 / (mse + eps))).mean().item()

@torch.no_grad()
def eval_dae(model, loader):
    model.eval(); mse_sum = ps_sum = n = 0
    for noisy, clean_, _ in loader:
        noisy, clean_ = noisy.to(device), clean_.to(device)
        rec = model(noisy)
        mse_sum += F.mse_loss(rec, clean_, reduction="sum").item()
        ps_sum  += psnr(rec, clean_) * clean_.size(0)
        n += clean_.size(0)
    return mse_sum / (n * 784), ps_sum / n


def train_dae(model, epochs=20, lr=2e-3, tag="DAE", loss_type="mse"):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    hist = {"train": [], "val_mse": [], "val_psnr": []}
    best, best_state = float("inf"), None
    for ep in range(1, epochs + 1):
        model.train(); tot = k = 0; t0 = time.time()
        for noisy, clean_, _ in tr_dl:
            noisy, clean_ = noisy.to(device, non_blocking=True), clean_.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(device.type == "cuda")):
                rec = model(noisy)
                loss = (F.mse_loss(rec, clean_) if loss_type == "mse"
                        else F.binary_cross_entropy(rec.clamp(1e-6, 1-1e-6), clean_))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * clean_.size(0); k += clean_.size(0)
        sched.step()
        vm, vp = eval_dae(model, va_dl)
        hist["train"].append(tot/k); hist["val_mse"].append(vm); hist["val_psnr"].append(vp)
        print(f"[{tag}] ep {ep:02d}/{epochs} | train {tot/k:.5f} | val MSE {vm:.5f} "
              f"| val PSNR {vp:.2f} dB | {time.time()-t0:.1f}s")
        if vm < best:
            best = vm
            best_state = {k2: v.detach().cpu().clone() for k2, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, hist

EPOCHS = 20
dense_dae, hist_dense = train_dae(DenseDAE(), epochs=EPOCHS, tag="DenseDAE")
print()
conv_dae,  hist_conv  = train_dae(ConvDAE(),  epochs=EPOCHS, tag="ConvDAE")

## 5. Qualitative results

In [ ]:
torch.manual_seed(7)
idx = np.random.choice(len(test_set), 10, replace=False)
clean_b = torch.stack([test_set[i][0] for i in idx])
torch.manual_seed(SEED)
noisy_b = add_noise(clean_b, NOISE_KIND, NOISE_LEVEL)

with torch.no_grad():
    rec_dense = dense_dae.eval()(noisy_b.to(device)).cpu()
    rec_conv  = conv_dae.eval()(noisy_b.to(device)).cpu()

rows = [("noisy input", noisy_b), ("DenseDAE", rec_dense), ("ConvDAE", rec_conv), ("clean target", clean_b)]
fig, axes = plt.subplots(len(rows), 10, figsize=(15, 6.5))
for r, (name, imgs) in enumerate(rows):
    for c in range(10):
        axes[r, c].imshow(imgs[c, 0], cmap="gray", vmin=0, vmax=1); axes[r, c].axis("off")
    axes[r, 0].set_title(name, loc="left", fontsize=10)
plt.suptitle(f"Denoising — {NOISE_KIND} noise, level {NOISE_LEVEL}")
plt.tight_layout(); plt.show()

## 6. Quantitative evaluation — MSE, PSNR, SSIM, and classical baselines

In [ ]:
from skimage.metrics import structural_similarity as ssim_fn
import scipy.ndimage as ndi

@torch.no_grad()
def full_metrics(fn, loader, limit=4000, name=""):
    mses, psnrs, ssims = [], [], []
    seen = 0
    for noisy, clean_, _ in loader:
        rec = fn(noisy)
        rec = rec.clamp(0, 1)
        m = F.mse_loss(rec, clean_, reduction="none").flatten(1).mean(1)
        mses.append(m)
        psnrs.append(10 * torch.log10(1.0 / (m + 1e-10)))
        r_np, c_np = rec.squeeze(1).numpy(), clean_.squeeze(1).numpy()
        for a, b in zip(r_np[:64], c_np[:64]):
            ssims.append(ssim_fn(b, a, data_range=1.0))
        seen += clean_.size(0)
        if seen >= limit:
            break
    return dict(name=name, MSE=torch.cat(mses).mean().item(),
                PSNR=torch.cat(psnrs).mean().item(), SSIM=float(np.mean(ssims)))

def torch_fn(model):
    def f(noisy):
        return model.eval()(noisy.to(device)).cpu()
    return f

def median_filter(noisy):
    a = noisy.squeeze(1).numpy()
    return torch.from_numpy(np.stack([ndi.median_filter(im, size=3) for im in a])).unsqueeze(1)

def gauss_filter(noisy):
    a = noisy.squeeze(1).numpy()
    return torch.from_numpy(np.stack([ndi.gaussian_filter(im, sigma=1.0) for im in a])).unsqueeze(1)

rows = [
    full_metrics(lambda n: n,            te_dl, name="no denoising (noisy)"),
    full_metrics(median_filter,          te_dl, name="median filter 3x3"),
    full_metrics(gauss_filter,           te_dl, name="gaussian filter s=1"),
    full_metrics(torch_fn(dense_dae),    te_dl, name="DenseDAE"),
    full_metrics(torch_fn(conv_dae),     te_dl, name="ConvDAE"),
]
print(f"{'method':<24}{'MSE':>10}{'PSNR (dB)':>12}{'SSIM':>9}")
print("-" * 55)
for r in rows:
    print(f"{r['name']:<24}{r['MSE']:>10.5f}{r['PSNR']:>12.2f}{r['SSIM']:>9.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
names = [r["name"] for r in rows]
for ax, key, ttl in zip(axes, ["MSE", "PSNR", "SSIM"], ["MSE (lower better)", "PSNR dB (higher better)", "SSIM (higher better)"]):
    ax.barh(names, [r[key] for r in rows], color="#4c8fbd")
    ax.set_title(ttl); ax.grid(axis="x", alpha=.3); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_dense["val_mse"], marker="o", ms=3, label="DenseDAE")
axes[0].plot(hist_conv["val_mse"],  marker="s", ms=3, label="ConvDAE")
axes[0].set_title("Validation MSE"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(hist_dense["val_psnr"], marker="o", ms=3, label="DenseDAE")
axes[1].plot(hist_conv["val_psnr"],  marker="s", ms=3, label="ConvDAE")
axes[1].set_title("Validation PSNR (dB)"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Generalisation across noise types and levels

In [ ]:
# The DAE was trained on gaussian(0.5) only. How does it cope with unseen corruptions?
print(f"{'noise':<14}{'level':>7}{'noisy PSNR':>13}{'ConvDAE PSNR':>15}{'gain (dB)':>11}")
print("-" * 60)
cross = {}
for kind in NOISE_KINDS:
    for lvl in [0.3, 0.5, 0.7]:
        dl = DataLoader(NoisyMNIST(test_set, kind, lvl, fixed=True, seed=5), batch_size=512, num_workers=2)
        base = full_metrics(lambda n: n, dl, limit=2000)
        out  = full_metrics(torch_fn(conv_dae), dl, limit=2000)
        cross[(kind, lvl)] = (base["PSNR"], out["PSNR"])
        print(f"{kind:<14}{lvl:>7.1f}{base['PSNR']:>13.2f}{out['PSNR']:>15.2f}"
              f"{out['PSNR']-base['PSNR']:>11.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
w = 0.12
for i, kind in enumerate(NOISE_KINDS):
    gains = [cross[(kind, l)][1] - cross[(kind, l)][0] for l in [0.3, 0.5, 0.7]]
    ax.bar(np.arange(3) + i * w, gains, w, label=kind)
ax.set_xticks(np.arange(3) + 2 * w); ax.set_xticklabels(["0.3", "0.5", "0.7"])
ax.set_xlabel("noise level"); ax.set_ylabel("PSNR gain (dB)")
ax.set_title("DAE trained on gaussian(0.5) — generalisation to other corruptions")
ax.axhline(0, color="k", lw=.8); ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Visual check on an UNSEEN corruption type
for kind in ["salt_pepper", "block"]:
    torch.manual_seed(3)
    nb = add_noise(clean_b, kind, 0.5)
    with torch.no_grad():
        rb = conv_dae.eval()(nb.to(device)).cpu()
    fig, axes = plt.subplots(3, 10, figsize=(15, 5))
    for c in range(10):
        axes[0, c].imshow(nb[c, 0], cmap="gray", vmin=0, vmax=1); axes[0, c].axis("off")
        axes[1, c].imshow(rb[c, 0], cmap="gray", vmin=0, vmax=1); axes[1, c].axis("off")
        axes[2, c].imshow(clean_b[c, 0], cmap="gray", vmin=0, vmax=1); axes[2, c].axis("off")
    for r, nm in enumerate([f"{kind} (unseen)", "ConvDAE output", "clean"]):
        axes[r, 0].set_title(nm, loc="left", fontsize=9)
    plt.tight_layout(); plt.show()

## 8. Bottleneck sweep — how much compression is too much?

In [ ]:
SWEEP_EPOCHS = 8
latent_sizes = [4, 8, 16, 32, 64, 128]
sweep = {}
for L in latent_sizes:
    m, _ = train_dae(DenseDAE(latent=L), epochs=SWEEP_EPOCHS, tag=f"latent={L}")
    mse, ps = eval_dae(m, te_dl)
    sweep[L] = (mse, ps)
    print(f"  latent {L:>4} -> test MSE {mse:.5f} | PSNR {ps:.2f} dB\n")

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(latent_sizes, [sweep[l][0] for l in latent_sizes], "o-", color="#1f77b4")
ax1.set_xscale("log", base=2); ax1.set_xlabel("bottleneck size (latent units)")
ax1.set_ylabel("test MSE", color="#1f77b4"); ax1.grid(alpha=.3)
ax2 = ax1.twinx()
ax2.plot(latent_sizes, [sweep[l][1] for l in latent_sizes], "s--", color="#d62728")
ax2.set_ylabel("test PSNR (dB)", color="#d62728")
plt.title("Reconstruction quality vs bottleneck width")
plt.tight_layout(); plt.show()
print("Too narrow -> underfitting (digits blur together). Too wide -> the DAE starts")
print("copying the noise through instead of learning the manifold.")

## 9. What does the latent space look like?

In [ ]:
from sklearn.manifold import TSNE

Z, Y = [], []
conv_dae.eval()
with torch.no_grad():
    for noisy, clean_, lab in te_dl:
        Z.append(conv_dae.encode(noisy.to(device)).cpu()); Y.append(lab)
        if sum(z.shape[0] for z in Z) >= 3000:
            break
Z = torch.cat(Z)[:3000].numpy(); Y = torch.cat(Y)[:3000].numpy()
print("latent dim:", Z.shape[1])

emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(Z)
plt.figure(figsize=(6.8, 5.8))
sc = plt.scatter(emb[:, 0], emb[:, 1], c=Y, cmap="tab10", s=6, alpha=.7)
plt.colorbar(sc, ticks=range(10), label="digit")
plt.title("t-SNE of the ConvDAE latent code (encoded from NOISY inputs)")
plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()
print("Digit clusters emerge although the encoder never saw a label and never saw a clean image")
print("-> the denoising objective alone recovers class structure. That is the CO5 takeaway.")

In [ ]:
# Latent interpolation: walk between two digits in latent space
a_idx, b_idx = 3, 2
xa = test_set[np.where(np.array(test_set.targets) == 3)[0][0]][0]
xb_ = test_set[np.where(np.array(test_set.targets) == 8)[0][0]][0]
with torch.no_grad():
    za = conv_dae.encoder(xa.unsqueeze(0).to(device))
    zb = conv_dae.encoder(xb_.unsqueeze(0).to(device))
    steps = torch.linspace(0, 1, 10, device=device).view(-1, 1, 1, 1)
    zi = (1 - steps) * za + steps * zb
    out = conv_dae.decoder(zi).cpu()

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(out[i, 0], cmap="gray", vmin=0, vmax=1); ax.axis("off")
plt.suptitle("Latent interpolation:  3  ->  8   (the decoder produces valid-looking digits throughout)")
plt.tight_layout(); plt.show()

## 10. Downstream utility — does denoising actually help a classifier?

PSNR is a proxy. The real question for a production pipeline is whether the DAE **recovers task
accuracy** on corrupted inputs. We train a small CNN classifier on *clean* MNIST, then test it on
(a) clean, (b) noisy, (c) DAE-denoised images.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(64, 10))
    def forward(self, x):
        return self.net(x)

clf = SmallCNN().to(device)
clf_dl = DataLoader(train_set, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
opt = torch.optim.AdamW(clf.parameters(), lr=2e-3, weight_decay=1e-4)
for ep in range(1, 4):
    clf.train(); c = n = 0
    for x, y in clf_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        out = clf(x); loss = F.cross_entropy(out, y); loss.backward(); opt.step()
        c += (out.argmax(1) == y).sum().item(); n += y.size(0)
    print(f"[classifier] ep {ep}/3 train acc {c/n:.4f}")

@torch.no_grad()
def clf_acc(transform=None):
    clf.eval(); c = n = 0
    for noisy, clean_, lab in te_dl:
        x = clean_ if transform == "clean" else noisy
        x = x.to(device)
        if transform == "dae":
            x = conv_dae.eval()(x)
        c += (clf(x).argmax(1).cpu() == lab).sum().item(); n += lab.size(0)
    return c / n

a_clean = clf_acc("clean"); a_noisy = clf_acc(None); a_dae = clf_acc("dae")
print(f"\nclassifier accuracy on CLEAN      images: {a_clean:.4f}")
print(f"classifier accuracy on NOISY      images: {a_noisy:.4f}   (drop {a_clean-a_noisy:.4f})")
print(f"classifier accuracy on DENOISED   images: {a_dae:.4f}   (recovered "
      f"{(a_dae-a_noisy)/max(a_clean-a_noisy,1e-9):.1%} of the lost accuracy)")

plt.figure(figsize=(5.4, 3.6))
plt.bar(["clean", "noisy", "DAE-denoised"], [a_clean, a_noisy, a_dae],
        color=["#2ca02c", "#d62728", "#1f77b4"])
plt.ylim(0, 1.02); plt.ylabel("classifier accuracy")
plt.title("Downstream utility of denoising"); plt.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

## 11. Bonus — anomaly detection with reconstruction error

An autoencoder trained on MNIST reconstructs digits well and **non-digits badly**. That
reconstruction error is a ready-made anomaly score — a standard industrial use of autoencoders.

In [ ]:
# Anomalies = random noise images + inverted digits + FashionMNIST (if available)
normal = torch.stack([test_set[i][0] for i in range(500)])
anom_list = [torch.rand(150, 1, 28, 28), 1.0 - torch.stack([test_set[i][0] for i in range(150)])]
try:
    fm = datasets.FashionMNIST("./data", train=False, download=True, transform=transforms.ToTensor())
    anom_list.append(torch.stack([fm[i][0] for i in range(200)]))
    print("FashionMNIST included as an out-of-distribution source")
except Exception as e:
    print("FashionMNIST unavailable, using noise + inverted digits only")
anom = torch.cat(anom_list)

@torch.no_grad()
def recon_error(x):
    x = x.to(device)
    r = conv_dae.eval()(x)
    return F.mse_loss(r, x, reduction="none").flatten(1).mean(1).cpu().numpy()

e_norm, e_anom = recon_error(normal), recon_error(anom)

from sklearn.metrics import roc_auc_score, roc_curve
scores = np.concatenate([e_norm, e_anom])
truth  = np.concatenate([np.zeros(len(e_norm)), np.ones(len(e_anom))])
auc = roc_auc_score(truth, scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(e_norm, bins=40, alpha=.7, label="normal (MNIST digits)", density=True)
axes[0].hist(e_anom, bins=40, alpha=.7, label="anomalous", density=True)
axes[0].set_xlabel("reconstruction MSE"); axes[0].legend(); axes[0].set_title("Reconstruction-error distributions")
fpr, tpr, _ = roc_curve(truth, scores)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}"); axes[1].plot([0, 1], [0, 1], "k--", lw=.8)
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].set_title("Anomaly detection ROC")
axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Anomaly detection ROC-AUC: {auc:.4f}")

In [ ]:
torch.save({"conv_dae": conv_dae.state_dict(), "dense_dae": dense_dae.state_dict(),
            "noise_kind": NOISE_KIND, "noise_level": NOISE_LEVEL}, "denoising_autoencoder.pth")
print("saved denoising_autoencoder.pth")

## 12. Conclusion

| Method | MSE | PSNR | SSIM | Comment |
|---|---|---|---|---|
| No denoising | highest | lowest | lowest | the corrupted input |
| Median / Gaussian filter | moderate | moderate | moderate | no learning — removes noise *and* stroke detail |
| DenseDAE | low | high | high | learns the manifold, but ignores spatial structure |
| **ConvDAE** | **lowest** | **highest** | **highest** | weight sharing + spatial priors win clearly |

**Key findings**

1. **The corruption is the supervision.** Training `x̃ -> x` stops the autoencoder from learning
   the identity map; the noise is a regularizer, not an obstacle.
2. **Convolutional beats dense** at similar or lower parameter count, because MNIST digits are
   spatially local and translation-equivariant.
3. **Learned denoising beats classical filters** by a clear margin on both PSNR and SSIM — the
   filters cannot distinguish a stroke from a speck; the DAE can, because it knows what a digit is.
4. **Bottleneck width is a real hyperparameter:** too narrow and reconstructions blur across
   classes; too wide and noise leaks through the code.
5. **Generalisation is limited by the training corruption.** The DAE trained on Gaussian noise
   still gains PSNR on speckle and blur, but helps much less on block masking — the fix is to
   randomise the corruption type during training (try it: sample `kind` per batch).
6. **Denoising restores downstream accuracy**, recovering most of the classifier accuracy lost to
   noise — the metric that actually matters in a pipeline.
7. The same model doubles as an **anomaly detector** via reconstruction error.

**Extensions:** train on a random mixture of corruption types; add skip connections (U-Net) for
sharper edges; swap MSE for a perceptual or SSIM loss to avoid over-smoothing; extend to a
**variational** autoencoder for generation; scale up to CIFAR-10 or real medical images.